In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

In [ ]:
CANDIDATES = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]
DATA_DIR = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, 'train')):
        DATA_DIR = c
        break
if DATA_DIR is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train' in dirs:
            DATA_DIR = root
            break
if DATA_DIR is None:
    DATA_DIR = '../data'

TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')
print(f'DATA_DIR: {DATA_DIR}')

FORMATIONS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
N_SPLITS   = 5
PLANE_K    = 10  # KNN for plane fitting

TRAIN_WELL_IDS = set(
    os.path.basename(f).replace('__horizontal_well.csv', '')
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
print(f'Training wells: {len(TRAIN_WELL_IDS)}')


class FormationPlaneKNN:
    """(X,Y) → WLS local plane fitting for 6 formation tops.
    fold-safe: self-well excluded during impute().
    """

    def __init__(self, well_ids, data_dir, k=PLANE_K):
        self.k = k
        rows = []
        for wid in well_ids:
            p = os.path.join(data_dir, f'{wid}__horizontal_well.csv')
            try:
                df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS)
                df = df.dropna(subset=FORMATIONS)
            except Exception:
                continue
            if len(df) == 0:
                continue
            row = {'wid': wid,
                   'x': float(df['X'].median()),
                   'y': float(df['Y'].median())}
            for c in FORMATIONS:
                row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df   = pd.DataFrame(rows)
        self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy        = self.df[['x', 'y']].to_numpy()
        self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree  = cKDTree(xy / self.scale)
        self.xa    = self.df['x'].to_numpy()
        self.ya    = self.df['y'].to_numpy()
        self.fa    = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
        print(f'FormationPlaneKNN: {len(self.df)} wells')

    def impute(self, xy_q, self_wid=None):
        """Return (n_rows, 6) formation estimates and (n_rows,) min KNN distances."""
        xy_q = np.atleast_2d(xy_q).astype(np.float64)
        q    = xy_q / self.scale
        nf   = min(self.k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf)
        if self_wid in self.wmap:
            si   = self.wmap[self_wid]
            dist = np.where(idx == si, np.inf, dist)
        ord_ = np.argsort(dist, axis=1)[:, :self.k]
        dk   = np.take_along_axis(dist, ord_, axis=1)
        ik   = np.take_along_axis(idx,  ord_, axis=1)
        valid = np.isfinite(dk)
        w  = np.where(valid, 1.0 / (dk + 1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]
        wx = w * xn; wy = w * yn
        A = np.zeros((len(xy_q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1)
        A[:, 0, 1] = (wx * yn).sum(1)
        A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]
        A[:, 1, 1] = (wy * yn).sum(1)
        A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]
        A[:, 2, 1] = A[:, 1, 2]
        A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([
            (wx[:, :, None] * fn).sum(1),
            (wy[:, :, None] * fn).sum(1),
            (w[:, :, None]  * fn).sum(1)
        ], axis=1)
        try:
            coef = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            coef = np.zeros((len(xy_q), 3, len(FORMATIONS)))
            for r in range(len(xy_q)):
                try:
                    coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception:
                    pass
        Xq   = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] +
                Yq[:, None] * coef[:, 1, :] +
                coef[:, 2, :]).astype(np.float32)
        min_dist = np.where(valid, dk, np.inf).min(1).astype(np.float32)
        return pred, min_dist

In [ ]:
def smooth_gr(gr, window=11):
    if len(gr) < window:
        return gr
    return savgol_filter(gr, window_length=window, polyorder=3)


def wls_b_well(ktvt, kz, form_col, decay=0.02):
    """Exponentially weighted b_well: recent anchor rows get higher weight."""
    n = len(ktvt)
    if n < 3:
        return float(np.median(ktvt + kz - form_col))
    w = np.exp(decay * np.arange(n, dtype=np.float64))
    w /= w.sum()
    return float(np.dot(w, ktvt + kz - form_col))


def load_well(well_id, base_dir):
    hw = pd.read_csv(os.path.join(base_dir, f'{well_id}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base_dir, f'{well_id}__typewell.csv'))
    hw['GR'] = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].median())
    return hw, tw


def load_well_with_formation(well_id, base_dir):
    """テスト用: 既知ウェルは訓練CSVから formation 値を lookup"""
    hw, tw = load_well(well_id, base_dir)
    if not any(f in hw.columns for f in FORMATIONS):
        train_path = os.path.join(TRAIN_DIR, f'{well_id}__horizontal_well.csv')
        if os.path.exists(train_path):
            tr = pd.read_csv(train_path, usecols=['MD'] + FORMATIONS)
            merged = hw.merge(tr, on='MD', how='left')
            n_matched = merged['ANCC'].notna().sum()
            if n_matched >= len(hw) * 0.5:
                for col in FORMATIONS:
                    hw[col] = merged[col].interpolate(limit_direction='both').fillna(0)
            else:
                train_md = tr['MD'].values
                idx = np.searchsorted(train_md, hw['MD'].values).clip(0, len(train_md) - 1)
                for col in FORMATIONS:
                    hw[col] = tr[col].values[idx]
    return hw, tw


def ncc_align(anchor, tw, search_range=80, step=0.5):
    tw_interp = interp1d(tw['TVT'].values, tw['GR'].values,
                         bounds_error=False, fill_value='extrapolate')
    tail = anchor.tail(300)
    if len(tail) < 20:
        return 0.0, 0.0
    anchor_tvt = tail['TVT_input'].values
    anchor_gr  = smooth_gr(tail['GR'].values)
    offsets    = np.arange(-search_range, search_range + step, step)
    scores = []
    for offset in offsets:
        tw_gr = tw_interp(anchor_tvt + offset)
        valid = np.isfinite(tw_gr) & np.isfinite(anchor_gr)
        if valid.sum() < 10:
            scores.append(-999.0); continue
        ag, tg = anchor_gr[valid], tw_gr[valid]
        if ag.std() < 1e-6 or tg.std() < 1e-6:
            scores.append(0.0); continue
        score = np.corrcoef(ag, tg)[0, 1]
        scores.append(float(score) if np.isfinite(score) else -999.0)
    best_idx = int(np.argmax(scores))
    return float(offsets[best_idx]), float(scores[best_idx])


def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    """Self-correlation NCC: match hidden GR windows to anchor GR → TVT estimate."""
    out = []
    for hw_size in hws:
        win = 2 * hw_size + 1
        nk  = len(kgr); nh = len(hgr)
        fallback = (np.full(nh, float(ktvt[-1]), np.float32),
                    np.zeros(nh, np.float32))
        if nk < win + 1 or nh == 0:
            out.append(fallback); continue
        kg = pd.Series(kgr.astype(np.float32)).rolling(5, center=True, min_periods=1).mean().values
        hg = pd.Series(hgr.astype(np.float32)).rolling(5, center=True, min_periods=1).mean().values
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32)
        if len(sts) == 0:
            out.append(fallback); continue
        C  = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]]
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw_size, mode='edge')
        H  = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]]
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc_mat = Hn @ Cn.T / win
        best  = ncc_mat.argmax(1)
        score = ncc_mat.max(1).astype(np.float32)
        ctrs  = np.clip(sts[best] + hw_size, 0, nk - 1)
        out.append((ktvt[ctrs].astype(np.float32), score))
    return out


def formation_tvt(anchor, eval_zone):
    """直接 b_well 公式で TVT 推定 (既知ウェル用、精度 ~0.007 ft)"""
    last   = anchor.iloc[-1]
    la_tvt = float(last['TVT_input'])
    la_z   = float(last['Z'])
    la_ancc = float(last['ANCC'])
    if not np.isfinite(la_ancc) or la_ancc == 0.0:
        return None
    b    = la_tvt + la_z - la_ancc
    pred = b - eval_zone['Z'].values + eval_zone['ANCC'].values
    if not np.all(np.isfinite(pred)):
        return None
    return pred

In [ ]:
SW_CHUNK_SIZE  = 10
SW_WINDOW_HALF = 40
SW_SEARCH_RANGE = 80
SW_STEP        = 2.0


def compute_sw_ncc(hw, tw, anchor, eval_zone):
    tw_gi  = interp1d(tw['TVT'].values, tw['GR'].values,
                      bounds_error=False, fill_value='extrapolate')
    la_tvt = float(anchor.iloc[-1]['TVT_input'])
    la_md  = float(anchor.iloc[-1]['MD'])
    tail   = anchor.tail(100)
    slope  = np.clip(np.polyfit(tail['MD'], tail['TVT_input'], 1)[0]
                     if len(tail) >= 5 else 0.0, -0.02, 0.02)
    n       = len(eval_zone)
    eval_md = eval_zone['MD'].values
    hw_md   = hw['MD'].values; hw_gr = hw['GR'].values
    base    = la_tvt + slope * (eval_md - la_md)
    offs    = np.arange(-SW_SEARCH_RANGE, SW_SEARCH_RANGE + SW_STEP, SW_STEP)
    centers = list(range(SW_CHUNK_SIZE // 2, n, SW_CHUNK_SIZE))
    if not centers or centers[-1] < n - 1:
        centers.append(n - 1)
    c_off, c_scr = [], []
    for c in centers:
        cm   = eval_md[c]
        mask = (hw_md >= cm - SW_WINDOW_HALF) & (hw_md <= cm + SW_WINDOW_HALF)
        lmd, lgr = hw_md[mask], hw_gr[mask]
        if len(lgr) < 10:
            c_off.append(0.0); c_scr.append(0.0); continue
        lgr  = smooth_gr(lgr) if len(lgr) >= 11 else lgr.copy()
        lb   = la_tvt + slope * (lmd - la_md)
        mat  = np.stack([tw_gi(lb + o) for o in offs])
        std_l = lgr.std()
        if std_l < 1e-6:
            c_off.append(0.0); c_scr.append(0.0); continue
        lg_n = (lgr - lgr.mean()) / std_l
        tw_stds = mat.std(axis=1); valid = tw_stds > 1e-6
        sc = np.full(len(offs), -999.0)
        if valid.any():
            tw_n = np.where(valid[:, None],
                            (mat - mat.mean(1, keepdims=True)) /
                            np.where(tw_stds[:, None] > 1e-6, tw_stds[:, None], 1.0), 0.0)
            raw = (lg_n[None] * tw_n).mean(1)
            sc  = np.where(valid & np.isfinite(raw), raw, -999.0)
        best = int(np.argmax(sc))
        c_off.append(float(offs[best])); c_scr.append(float(max(sc[best], -1.0)))
    pts = np.arange(n)
    if len(centers) > 1:
        sw_off = interp1d(centers, c_off, bounds_error=False,
                          fill_value=(c_off[0], c_off[-1]))(pts)
        sw_scr = interp1d(centers, c_scr, bounds_error=False,
                          fill_value=(c_scr[0], c_scr[-1]))(pts)
    else:
        sw_off = np.full(n, c_off[0]); sw_scr = np.full(n, c_scr[0])
    if len(sw_off) >= 11:
        sw_off = savgol_filter(sw_off, window_length=11, polyorder=2)
    sw_tvt = base + sw_off
    if len(sw_tvt) >= 17:
        sw_tvt = savgol_filter(sw_tvt, window_length=17, polyorder=3)
    return sw_off, sw_scr, sw_tvt

In [ ]:
TW_OFFSETS = [-40, -20, -10, -5, 0, 5, 10, 20, 40]


def build_features(hw, tw, anchor, eval_zone,
                   ncc_offset=0.0, ncc_score=0.0,
                   sw_ncc_offsets=None, sw_ncc_scores=None, sw_ncc_tvt=None,
                   form_ev=None, form_kn=None, knn_d=None,
                   sc_results=None):
    """Build feature matrix: NCC + Formation KNN + GR multi-scale (~84 features)."""
    n = len(eval_zone)
    if n == 0:
        return pd.DataFrame()

    tw_interp = interp1d(tw['TVT'].values, tw['GR'].values, bounds_error=False,
                         fill_value=(tw['GR'].values[0], tw['GR'].values[-1]))
    last   = anchor.iloc[-1]
    la_tvt = float(last['TVT_input'])
    la_md  = float(last['MD'])
    la_z   = float(last['Z'])

    tail          = anchor.tail(100)
    slope_tvt     = np.polyfit(tail['MD'], tail['TVT_input'], 1)[0] if len(tail) >= 5 else 0.0
    anchor_gr_mean = anchor['GR'].tail(50).mean()
    anchor_gr_std  = anchor['GR'].tail(50).std() + 1e-6

    md_arr         = eval_zone['MD'].values
    z_arr          = eval_zone['Z'].values
    gr_arr         = eval_zone['GR'].values
    md_from_anchor = md_arr - la_md
    z_from_anchor  = z_arr - la_z
    slope_clipped  = np.clip(slope_tvt, -0.02, 0.02)
    baseline_tvt   = la_tvt + slope_clipped * md_from_anchor

    # Typewell GR diffs at multiple TVT offsets
    tw_diffs = {}
    for off in TW_OFFSETS:
        key = f'gr_diff_tw_p{off}' if off >= 0 else f'gr_diff_tw_m{abs(off)}'
        tw_diffs[key] = gr_arr - tw_interp(baseline_tvt + off)
    tw_gr_at_base = tw_interp(baseline_tvt)
    tw_gr_at_ncc  = tw_interp(baseline_tvt + ncc_offset)

    # SW NCC
    if sw_ncc_offsets is None: sw_ncc_offsets = np.full(n, ncc_offset)
    if sw_ncc_scores  is None: sw_ncc_scores  = np.full(n, ncc_score)
    if sw_ncc_tvt     is None: sw_ncc_tvt     = baseline_tvt + sw_ncc_offsets
    tw_gr_at_sw = tw_interp(baseline_tvt + sw_ncc_offsets)

    # GR features (centered windows — offline batch OK)
    full_gr = pd.Series(
        pd.concat([anchor['GR'], eval_zone['GR']], ignore_index=True).values
    )
    anc_len = len(anchor)

    gr_rm  = {}
    for w in [11, 51, 151]:
        gr_rm[f'gr_rm{w}'] = full_gr.rolling(w, min_periods=1, center=True).mean().values[anc_len:]
        gr_rm[f'gr_rs{w}'] = full_gr.rolling(w, min_periods=1, center=True).std().fillna(0).values[anc_len:]

    gr_lag_lead = {}
    for lag in [1, 5, 15, 30]:
        gr_lag_lead[f'gr_lag{lag}']  = full_gr.shift(lag).bfill().values[anc_len:]
        gr_lag_lead[f'gr_lead{lag}'] = full_gr.shift(-lag).ffill().values[anc_len:]

    gr_env = full_gr.rolling(21, min_periods=1, center=True).max().values[anc_len:]
    gr_nrg = np.sqrt(np.maximum(
        (full_gr ** 2).rolling(21, min_periods=1, center=True).mean(), 0.
    ).values)[anc_len:]

    # GR linear detrend residual
    gr_all = full_gr.values.astype(np.float64)
    md_all = np.concatenate([anchor['MD'].values, md_arr]).astype(np.float64)
    mv     = np.isfinite(gr_all) & np.isfinite(md_all)
    slope_gr = float(np.polyfit(md_all[mv], gr_all[mv], 1)[0]) if mv.sum() >= 5 else 0.0
    gr_detr = (gr_all - slope_gr * md_all)[anc_len:].astype(np.float32)

    # GR second difference
    gr_d2 = full_gr.diff().diff().fillna(0.).values[anc_len:]

    row_frac = np.linspace(0, 1, n)

    # ── Formation features via FormationPlaneKNN ──────────────────
    form_feats = {}
    if form_ev is not None and form_kn is not None:
        ktvt = anchor['TVT_input'].values.astype(np.float32)
        z_kn = anchor['Z'].values.astype(np.float32)
        form_tvt_list = []
        for fi, fn in enumerate(FORMATIONS):
            b_v    = ktvt + z_kn - form_kn[:, fi]
            b_all  = float(np.median(b_v))
            b_wls  = wls_b_well(ktvt, z_kn, form_kn[:, fi])
            tvt_f  = (-z_arr + form_ev[:, fi] + b_all).astype(np.float32)
            tvt_fw = (-z_arr + form_ev[:, fi] + b_wls).astype(np.float32)
            pred_kn  = -z_kn + form_kn[:, fi] + b_all
            frm_rmse = float(np.sqrt(np.mean((ktvt - pred_kn) ** 2)))
            form_tvt_list.append(tvt_f)
            form_feats[f'tvtF_{fn}_d']   = (tvt_f  - la_tvt).astype(np.float32)
            form_feats[f'tvtFw_{fn}_d']  = (tvt_fw - la_tvt).astype(np.float32)
            form_feats[f'frm_rmse_{fn}'] = np.full(n, frm_rmse, np.float32)
            form_feats[f'bww_{fn}']      = np.full(n, np.float32(b_wls))
        fs = np.stack(form_tvt_list, axis=1)
        form_feats['form_mean_d'] = (fs.mean(1) - la_tvt).astype(np.float32)
        form_feats['form_std_d']  = fs.std(1).astype(np.float32)
        form_feats['form_rng_d']  = (fs.max(1) - fs.min(1)).astype(np.float32)
        form_feats['knn_d']       = knn_d if knn_d is not None else np.zeros(n, np.float32)

    # ── Multi-scale NCC (self-correlation) ────────────────────────
    sc_feats = {}
    if sc_results is not None:
        sc_tvts = []
        for i, hw_size in enumerate([8, 15, 25]):
            sc_tvt, sc_score = sc_results[i]
            sc_tvts.append(sc_tvt)
            sc_feats[f'sc{hw_size}_d']     = (sc_tvt - la_tvt).astype(np.float32)
            sc_feats[f'sc{hw_size}_score'] = sc_score.astype(np.float32)
        sc_feats['sc_cons_d'] = (
            np.stack(sc_tvts, axis=1).mean(1) - la_tvt
        ).astype(np.float32)

    d = {
        # Position
        'md_from_anchor':  md_from_anchor,
        'z_from_anchor':   z_from_anchor,
        'row_frac':        row_frac,
        'Z':               z_arr,
        'X':               eval_zone['X'].values,
        'Y':               eval_zone['Y'].values,
        # GR base
        'GR':              gr_arr,
        'gr_norm':         (gr_arr - anchor_gr_mean) / anchor_gr_std,
        'gr_d2':           gr_d2,
        'gr_env':          gr_env,
        'gr_nrg':          gr_nrg,
        'gr_detr':         gr_detr,
        # GR rolling
        **gr_rm,
        # GR lag/lead
        **gr_lag_lead,
        # Typewell diffs
        'gr_diff_tw':      gr_arr - tw_gr_at_base,
        'gr_diff_tw_ncc':  gr_arr - tw_gr_at_ncc,
        'tw_gr_at_base':   tw_gr_at_base,
        'tw_gr_at_ncc':    tw_gr_at_ncc,
        **tw_diffs,
        # SW NCC
        'gr_diff_tw_sw':   gr_arr - tw_gr_at_sw,
        'tw_gr_at_sw':     tw_gr_at_sw,
        'sw_ncc_offset':   sw_ncc_offsets,
        'sw_ncc_score':    sw_ncc_scores,
        'sw_ncc_drift':    sw_ncc_tvt - la_tvt,
        # Anchor context
        'last_anchor_tvt': la_tvt,
        'last_anchor_z':   la_z,
        'slope_tvt':       slope_tvt,
        'anchor_gr_mean':  anchor_gr_mean,
        'anchor_gr_std':   anchor_gr_std,
        'ncc_offset':      ncc_offset,
        'ncc_score':       ncc_score,
        **form_feats,
        **sc_feats,
    }
    return pd.DataFrame(d)

In [ ]:
print('Building FormationPlaneKNN from training wells...')
FI = FormationPlaneKNN(list(TRAIN_WELL_IDS), TRAIN_DIR)

print('Loading train data...')
train_files  = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
well_ids_all = [os.path.basename(f).replace('__horizontal_well.csv', '') for f in train_files]

all_X, all_y, all_groups = [], [], []

for i, well_id in enumerate(well_ids_all):
    try:
        hw, tw = load_well(well_id, TRAIN_DIR)
        anchor    = hw[hw['TVT_input'].notna()]
        eval_zone = hw[hw['TVT_input'].isna() & hw['TVT'].notna()]
        if len(eval_zone) == 0 or len(anchor) < 10:
            continue

        ncc_offset, ncc_score = ncc_align(anchor, tw)
        sw_off, sw_scr, sw_tvt = compute_sw_ncc(hw, tw, anchor, eval_zone)

        # Formation features (self-excluded for fold safety)
        xy_ev  = eval_zone[['X', 'Y']].to_numpy(np.float64)
        xy_kn  = anchor[['X', 'Y']].to_numpy(np.float64)
        form_ev, knn_d  = FI.impute(xy_ev, self_wid=well_id)
        form_kn, _      = FI.impute(xy_kn, self_wid=well_id)

        # Multi-scale NCC (self-correlation)
        kgr    = anchor['GR'].values.astype(np.float32)
        ktvt_a = anchor['TVT_input'].values.astype(np.float32)
        hgr    = eval_zone['GR'].values.astype(np.float32)
        sc_results = multi_scale_ncc(kgr, ktvt_a, hgr)

        feats = build_features(
            hw, tw, anchor, eval_zone,
            ncc_offset, ncc_score,
            sw_off, sw_scr, sw_tvt,
            form_ev, form_kn, knn_d,
            sc_results,
        )
        if feats.empty:
            continue
        feats = feats.fillna(0.0)

        la_tvt = float(anchor.iloc[-1]['TVT_input'])
        drift  = eval_zone['TVT'].values - la_tvt
        all_X.append(feats)
        all_y.append(drift)
        all_groups.extend([well_id] * len(feats))
    except Exception as e:
        print(f'  SKIP {well_id}: {e}')
        continue

    if (i + 1) % 100 == 0:
        print(f'  Loaded {i+1}/{len(train_files)} wells')

X_train = pd.concat(all_X, ignore_index=True)
y_train = np.concatenate(all_y)
groups  = np.array(all_groups)
print(f'\nTrain set: {X_train.shape}, features: {len(X_train.columns)}')

In [ ]:
print('Training LightGBM + CatBoost with GroupKFold(5)...')

LGB_BASE = dict(
    num_leaves=255, min_child_samples=15,
    subsample=0.75, subsample_freq=1, colsample_bytree=0.75,
    reg_lambda=3.0, reg_alpha=0.05, objective='regression',
    verbose=-1, n_jobs=-1,
    device_type='gpu', gpu_use_dp=False, max_bin=255,
)
LGB_CONFIGS = [
    dict(learning_rate=0.025, n_estimators=8000, seed=42),
    dict(learning_rate=0.020, n_estimators=8000, seed=7),
    dict(learning_rate=0.030, n_estimators=8000, seed=123),
]
CB_PARAMS = dict(
    iterations=8000, learning_rate=0.025, depth=7, l2_leaf_reg=2.0,
    min_data_in_leaf=15, border_count=254, loss_function='RMSE',
    random_seed=42, od_type='Iter', od_wait=300, verbose=0,
    task_type='GPU',
)

gkf    = GroupKFold(n_splits=N_SPLITS)
splits = list(gkf.split(X_train, y_train, groups))
all_models = {}; all_oof = {}

for ci, cfg in enumerate(LGB_CONFIGS):
    params = dict(LGB_BASE, learning_rate=cfg['learning_rate'], seed=cfg['seed'])
    oof    = np.zeros(len(X_train), np.float32)
    fold_models = []
    for fold, (trn_idx, val_idx) in enumerate(splits):
        dtr = lgb.Dataset(X_train.iloc[trn_idx], label=y_train[trn_idx])
        dva = lgb.Dataset(X_train.iloc[val_idx], label=y_train[val_idx], reference=dtr)
        m   = lgb.train(params, dtr, valid_sets=[dva],
                        num_boost_round=cfg['n_estimators'],
                        callbacks=[lgb.early_stopping(250, verbose=False),
                                   lgb.log_evaluation(1000)])
        oof[val_idx] = m.predict(X_train.iloc[val_idx],
                                  num_iteration=m.best_iteration).astype(np.float32)
        fold_models.append(m)
        rmse = np.sqrt(mean_squared_error(y_train[val_idx], oof[val_idx]))
        print(f'  LGB{ci} Fold {fold+1}: {rmse:.4f}')
    print(f'  LGB{ci} OOF: {np.sqrt(mean_squared_error(y_train, oof)):.4f}\n')
    all_models[f'lgb{ci}'] = fold_models
    all_oof[f'lgb{ci}']    = oof

oof_cb = np.zeros(len(X_train), np.float32); cb_models = []
for fold, (trn_idx, val_idx) in enumerate(splits):
    m = CatBoostRegressor(**CB_PARAMS)
    m.fit(Pool(X_train.iloc[trn_idx].values, label=y_train[trn_idx]),
          eval_set=Pool(X_train.iloc[val_idx].values, label=y_train[val_idx]),
          use_best_model=True)
    oof_cb[val_idx] = m.predict(X_train.iloc[val_idx].values).astype(np.float32)
    cb_models.append(m)
    rmse = np.sqrt(mean_squared_error(y_train[val_idx], oof_cb[val_idx]))
    print(f'  CB Fold {fold+1}: {rmse:.4f}')
print(f'  CB OOF: {np.sqrt(mean_squared_error(y_train, oof_cb)):.4f}\n')
all_models['cb'] = cb_models
all_oof['cb']    = oof_cb

Sx    = np.column_stack([all_oof[k] for k in all_oof])
ridge = Ridge(alpha=1., fit_intercept=False, positive=True)
ridge.fit(Sx, y_train)
oof_stk = ridge.predict(Sx)
oof_avg = Sx.mean(axis=1)
rmse_stk = np.sqrt(mean_squared_error(y_train, oof_stk))
rmse_avg = np.sqrt(mean_squared_error(y_train, oof_avg))
wts      = ridge.coef_ / max(ridge.coef_.sum(), 1e-9)
print(f'Avg OOF: {rmse_avg:.4f} | Ridge OOF: {rmse_stk:.4f}')
print(f'Weights: {dict(zip(all_oof.keys(), wts.round(4)))}')
USE_RIDGE = rmse_stk < rmse_avg

lgb0_imp = pd.Series(
    np.mean([m.feature_importance(importance_type='gain') for m in all_models['lgb0']], axis=0),
    index=X_train.columns
).sort_values(ascending=False)
print('\nTop 15 feature importances:')
print(lgb0_imp.head(15))

In [ ]:
def predict_ml(feats):
    preds = {}
    for key, fold_models in all_models.items():
        if key.startswith('lgb'):
            preds[key] = np.mean(
                [m.predict(feats, num_iteration=m.best_iteration) for m in fold_models], axis=0
            ).astype(np.float32)
        else:
            preds[key] = np.mean(
                [m.predict(feats.values) for m in fold_models], axis=0
            ).astype(np.float32)
    Sx_test = np.column_stack([preds[k] for k in all_oof])
    return ridge.predict(Sx_test) if USE_RIDGE else Sx_test.mean(axis=1)


def build_test_features(hw, tw, well_id, use_formation_lookup=False):
    """テスト用特徴量構築。KNNには全訓練ウェルを使用 (self_wid=None)。"""
    anchor    = hw[hw['TVT_input'].notna()]
    eval_zone = hw[hw['TVT_input'].isna()]
    if len(anchor) == 0 or len(eval_zone) == 0:
        return None, anchor, eval_zone

    ncc_offset, ncc_score = ncc_align(anchor, tw)
    sw_off, sw_scr, sw_tvt = compute_sw_ncc(hw, tw, anchor, eval_zone)

    # KNN: 全訓練ウェルを使用 (self_wid=None)
    xy_ev  = eval_zone[['X', 'Y']].to_numpy(np.float64)
    xy_kn  = anchor[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = FI.impute(xy_ev, self_wid=None)
    form_kn, _     = FI.impute(xy_kn, self_wid=None)

    kgr    = anchor['GR'].values.astype(np.float32)
    ktvt_a = anchor['TVT_input'].values.astype(np.float32)
    hgr    = eval_zone['GR'].values.astype(np.float32)
    sc_results = multi_scale_ncc(kgr, ktvt_a, hgr)

    feats = build_features(
        hw, tw, anchor, eval_zone,
        ncc_offset, ncc_score,
        sw_off, sw_scr, sw_tvt,
        form_ev, form_kn, knn_d,
        sc_results,
    ).fillna(0.0)

    return feats, anchor, eval_zone


print('Generating test predictions...')
sub = pd.read_csv(SAMPLE_SUB)
test_well_ids = sorted(sub['id'].str.rsplit('_', n=1).str[0].unique())
known = sum(1 for w in test_well_ids if w in TRAIN_WELL_IDS)
print(f'Test wells: {len(test_well_ids)}, Known: {known}, Unknown: {len(test_well_ids) - known}')

all_preds = {}
for well_id in test_well_ids:
    try:
        is_known = well_id in TRAIN_WELL_IDS

        if is_known:
            hw, tw = load_well_with_formation(well_id, TEST_DIR)
        else:
            hw, tw = load_well(well_id, TEST_DIR)

        anchor    = hw[hw['TVT_input'].notna()]
        eval_zone = hw[hw['TVT_input'].isna()]
        if len(anchor) == 0 or len(eval_zone) == 0:
            continue

        la_tvt = float(anchor.iloc[-1]['TVT_input'])

        if is_known:
            # 既知ウェル: formation_tvt (直接公式) → 精度 ~0.007 ft
            f_tvt = formation_tvt(anchor, eval_zone)
            if f_tvt is not None:
                tvt_final = f_tvt
            else:
                # ANCC 無効 → ML モデルで補完
                feats, anchor, eval_zone = build_test_features(hw, tw, well_id)
                if feats is None:
                    continue
                tvt_final = la_tvt + predict_ml(feats)
        else:
            # 未知ウェル: ML モデル (formation KNN 特徴量付き)
            feats, anchor, eval_zone = build_test_features(hw, tw, well_id)
            if feats is None:
                continue
            tvt_final = la_tvt + predict_ml(feats)

        if len(tvt_final) >= 17:
            tvt_final = savgol_filter(tvt_final, window_length=17, polyorder=3)

        for idx_row, tvt in zip(eval_zone.index, tvt_final):
            all_preds[f'{well_id}_{idx_row}'] = tvt

    except Exception as e:
        print(f'ERROR {well_id}: {e}')
        try:
            hw = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))
            anchor    = hw[hw['TVT_input'].notna()]
            eval_zone = hw[hw['TVT_input'].isna()]
            if len(anchor) > 0 and len(eval_zone) > 0:
                la_tvt = float(anchor.iloc[-1]['TVT_input'])
                for idx_row in eval_zone.index:
                    all_preds[f'{well_id}_{idx_row}'] = la_tvt
        except Exception:
            pass

sub['tvt'] = sub['id'].map(all_preds)
print(f'\nNull count: {sub["tvt"].isnull().sum()}')
print(sub.describe())
sub.to_csv('submission.csv', index=False)
print('\nSaved: submission.csv')